In [9]:
# ======================================
# IMPORTS
# ======================================
import re
from collections import defaultdict
from datasets import load_dataset

# ======================================
# LOAD DATA (WikiText-2)
# ======================================
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
text = " ".join(dataset['train']['text'])

# ======================================
# PREPROCESSING
# ======================================
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # keep only letters + space
    tokens = text.split()
    return tokens

tokens = preprocess(text)

# ======================================
#  BUILD N-GRAMS (TRIGRAM: k=2 context)
# ======================================
n = 3  # trigram
context_counts = defaultdict(int)
ngram_counts = defaultdict(int)

for i in range(len(tokens) - n + 1):
    context = tuple(tokens[i:i+n-1])   # (w1, w2)
    target = tokens[i+n-1]             # w3

    context_counts[context] += 1
    ngram_counts[(context, target)] += 1

# ======================================
#  VOCAB
# ======================================
vocab = set(tokens)
V = len(vocab)

# ======================================
# PROBABILITY (LAPLACE SMOOTHING)
# ======================================
def get_prob(context, word):
    context = tuple(context)
    if context not in context_counts:
        return 1 / V   # unseen context → uniform probability
    return (ngram_counts[(context, word)] + 1) / (context_counts[context] + V)

# ======================================
# PREDICT NEXT WORD
# ======================================
def predict_next(context):
    context = tuple(context)

    best_word = None
    best_prob = -1

    for word in vocab:
        prob = get_prob(context, word)
        if prob > best_prob:
            best_prob = prob
            best_word = word

    return best_word

# ======================================
# TOP-K PREDICTIONS (BETTER OUTPUT)
# ======================================
def top_k_predictions(context, k=5):
    context = tuple(context)

    probs = []
    for word in vocab:
        probs.append((word, get_prob(context, word)))

    probs.sort(key=lambda x: x[1], reverse=True)
    return probs[:k]

# ======================================
# TEST
# ======================================
print("Next word prediction:")
print("Context: ['the', 'king'] →", predict_next(["the", "king"]))
print("Context: ['i', 'am'] →", predict_next(["i", "am"]))

print("\nTop-5 predictions:")
print(top_k_predictions(["the", "king"], 5))

Next word prediction:
Context: ['the', 'king'] → s
Context: ['i', 'am'] → not

Top-5 predictions:
[('s', 0.000960261191043964), ('of', 0.0003200870636813213), ('and', 0.00020805659139285887), ('in', 0.00014403917865659459), ('the', 0.00011203047228846246)]


In [10]:
import random

def sample_next_topk(context, k=5):
    context = tuple(context)

    candidates = []

    # collect only valid next words
    for (ctx, word), count in ngram_counts.items():
        if ctx == context:
            prob = get_prob(context, word)
            candidates.append((word, prob))

    # fallback
    if not candidates:
        return random.choice(list(vocab))

    #  Step 1: sort by probability
    candidates.sort(key=lambda x: x[1], reverse=True)

    # Step 2: take top K
    top_k = candidates[:k]

    words = [w for w, _ in top_k]
    probs = [p for _, p in top_k]

    # step 3: normalize
    total = sum(probs)
    probs = [p / total for p in probs]

    #Step 4: CDF sampling
    r = random.random()
    cumulative = 0

    print("\n--- Top-K Candidates ---")
    for w, p in zip(words, probs):
        cumulative += p
        print(f"{w:15s} | Prob: {p:.4f} | CDF: {cumulative:.4f}")

        if r <= cumulative:
            print(f"\nRandom r = {r:.4f}")
            print(f"Selected: {w}")
            return w

    return words[-1]

In [11]:
sample_next_topk(["the", "king"], k=5)
sample_next_topk(["the", "king"], k=5)
sample_next_topk(["the", "king"], k=5)


--- Top-K Candidates ---
s               | Prob: 0.5505 | CDF: 0.5505

Random r = 0.0764
Selected: s

--- Top-K Candidates ---
s               | Prob: 0.5505 | CDF: 0.5505

Random r = 0.5316
Selected: s

--- Top-K Candidates ---
s               | Prob: 0.5505 | CDF: 0.5505
of              | Prob: 0.1835 | CDF: 0.7339
and             | Prob: 0.1193 | CDF: 0.8532
in              | Prob: 0.0826 | CDF: 0.9358

Random r = 0.9197
Selected: in


'in'

In [12]:
import random
import numpy as np

def sample_next_topp_temp(context, p=0.8, T=1.0, debug=True):
    context = tuple(context)

    candidates = []

    # Step 1: collect candidates
    for (ctx, word), count in ngram_counts.items():
        if ctx == context:
            prob = get_prob(context, word)
            candidates.append((word, prob))

    # fallback
    if not candidates:
        return random.choice(list(vocab))

    # Step 2: apply temperature
    words = []
    probs = []

    for word, prob in candidates:
        words.append(word)
        probs.append(prob)

    probs = np.array(probs)

    #Temperature scaling
    probs = probs ** (1 / T)

    # normalize
    probs = probs / probs.sum()

    # Step 3: sort by probability
    sorted_indices = np.argsort(probs)[::-1]

    words = [words[i] for i in sorted_indices]
    probs = probs[sorted_indices]

    # Step 4: Top-P filtering
    selected_words = []
    selected_probs = []

    cumulative = 0

    for w, pr in zip(words, probs):
        selected_words.append(w)
        selected_probs.append(pr)
        cumulative += pr
        if cumulative >= p:
            break

    # Step 5: normalize again
    selected_probs = np.array(selected_probs)
    selected_probs = selected_probs / selected_probs.sum()

    # Step 6: CDF sampling
    r = random.random()
    cumulative = 0

    if debug:
        print("\n--- Top-P + Temperature Sampling ---")
        print(f"Temperature (T): {T}, Top-P: {p}")

    for w, pr in zip(selected_words, selected_probs):
        cumulative += pr

        if debug:
            print(f"{w:15s} | Prob: {pr:.4f} | CDF: {cumulative:.4f}")

        if r <= cumulative:
            if debug:
                print(f"\nRandom r = {r:.4f}")
                print(f"Selected: {w}")
            return w

    return selected_words[-1]

In [13]:
sample_next_topp_temp(["the", "king"], p=0.8, T=1.0)
sample_next_topp_temp(["the", "king"], p=0.8, T=0.7)
sample_next_topp_temp(["the", "king"], p=0.9, T=1.5)

Streaming output truncated to the last 5000 lines.
fluorite        | Prob: 0.0000 | CDF: 0.1472
bauldie         | Prob: 0.0000 | CDF: 0.1472
ballsy          | Prob: 0.0000 | CDF: 0.1472
barometer       | Prob: 0.0000 | CDF: 0.1473
deserted        | Prob: 0.0000 | CDF: 0.1473
aashto          | Prob: 0.0000 | CDF: 0.1473
whalebone       | Prob: 0.0000 | CDF: 0.1473
dinners         | Prob: 0.0000 | CDF: 0.1473
best            | Prob: 0.0000 | CDF: 0.1473
katakacharya    | Prob: 0.0000 | CDF: 0.1474
plated          | Prob: 0.0000 | CDF: 0.1474
lg              | Prob: 0.0000 | CDF: 0.1474
masterpiece     | Prob: 0.0000 | CDF: 0.1474
selectors       | Prob: 0.0000 | CDF: 0.1474
testified       | Prob: 0.0000 | CDF: 0.1475
hamarwein       | Prob: 0.0000 | CDF: 0.1475
nostrovite      | Prob: 0.0000 | CDF: 0.1475
overheating     | Prob: 0.0000 | CDF: 0.1475
speech          | Prob: 0.0000 | CDF: 0.1475
nations         | Prob: 0.0000 | CDF: 0.1475
daunorubicin    | Prob: 0.0000 | CDF: 0.1476
mast

'radiogenic'